# System Metrics Monitoring

When running latency benchmarks — especially load tests with high concurrency — it's important to know whether the **client machine itself** is a bottleneck. High CPU usage, memory pressure, or network saturation on the client side can skew latency measurements without any visible errors.

LLMeter's `SystemMetricsMonitor` callback tracks CPU, memory, and network I/O during benchmark runs using [psutil](https://github.com/giampaolo/psutil). This notebook demonstrates how to:

1. Attach system metrics monitoring to a Runner
2. Inspect aggregated statistics after a run
3. Access raw time-series samples for custom analysis
4. Correlate system resource usage with latency across different concurrency levels
5. Detect client-side bottlenecks

## Setup

Install LLMeter with the `system-metrics` extra (which brings in `psutil`):

In [ ]:
%pip install "llmeter[system-metrics,plotting]<1"

In [ ]:
from llmeter.callbacks.system_metrics import SystemMetricsMonitor
from llmeter.endpoints.bedrock import BedrockConverseStream
from llmeter.runner import Runner

This notebook assumes you have [configured AWS credentials](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/credentials.html) with `bedrock:InvokeModel` and `bedrock:InvokeModelWithResponseStream` permissions, and that you've [enabled access](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html) to the model below.

In [ ]:
model_id = None  # <-- Set a Bedrock model ID, e.g. "us.anthropic.claude-3-5-haiku-20241022-v1:0"

if not model_id:
    raise ValueError("Please set a valid model ID above!")

In [ ]:
endpoint = BedrockConverseStream(model_id=model_id)

## Basic Usage

Create a `SystemMetricsMonitor` and attach it to a `Runner`. The monitor will automatically start sampling when the run begins and stop when it ends.

In [ ]:
monitor = SystemMetricsMonitor(sample_interval=0.5, per_process=True)

runner = Runner(
    endpoint=endpoint,
    callbacks=[monitor],
    output_path="outputs/system-metrics",
)

payload = endpoint.create_payload(
    "Write a short haiku about cloud computing.",
    max_tokens=100,
)

In [ ]:
result = await runner.run(payload=payload, clients=5, n_requests=10)

## Inspecting Aggregated Statistics

After the run, system metrics are available in `result.stats` alongside the usual latency and throughput metrics:

In [ ]:
# CPU usage during the run
print(f"CPU average: {result.stats['system_cpu_percent-average']:.1f}%")
print(f"CPU p90:     {result.stats['system_cpu_percent-p90']:.1f}%")
print(f"CPU p99:     {result.stats['system_cpu_percent-p99']:.1f}%")
print()

# Memory usage
print(f"Memory RSS average: {result.stats['system_memory_rss_mb-average']:.1f} MB")
print(f"Memory RSS peak:    {result.stats['system_memory_rss_mb-max']:.1f} MB")
print()

# Network I/O
print(f"Network sent:     {result.stats['system_net_bytes_sent_total']:,} bytes")
print(f"Network received: {result.stats['system_net_bytes_recv_total']:,} bytes")
print()

# How many samples were collected
print(f"Samples collected: {result.stats['system_samples_collected']}")

## Accessing Raw Samples

For time-series analysis, you can access the raw samples collected during the run. Each sample includes a timestamp, CPU percentage, memory (RSS and VMS), and cumulative network counters.

In [ ]:
import pandas as pd

# Convert samples to a DataFrame for easy analysis
samples_df = pd.DataFrame(
    [
        {
            "time": s.timestamp - monitor.samples[0].timestamp,  # relative time
            "cpu_percent": s.cpu_percent,
            "memory_rss_mb": s.memory_rss_mb,
            "net_bytes_recv": s.net_bytes_recv,
        }
        for s in monitor.samples
    ]
)

samples_df.head(10)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=("CPU Usage (%)", "Memory RSS (MB)", "Network Received (bytes)"),
    vertical_spacing=0.08,
)

fig.add_trace(
    go.Scatter(x=samples_df["time"], y=samples_df["cpu_percent"], mode="lines+markers", name="CPU %"),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=samples_df["time"], y=samples_df["memory_rss_mb"], mode="lines+markers", name="RSS MB"),
    row=2, col=1,
)
fig.add_trace(
    go.Scatter(x=samples_df["time"], y=samples_df["net_bytes_recv"], mode="lines+markers", name="Net Recv"),
    row=3, col=1,
)

fig.update_layout(height=700, title_text="System Metrics Over Time", showlegend=False)
fig.update_xaxes(title_text="Time (seconds)", row=3, col=1)
fig.show()

## Correlating System Metrics with Concurrency

A common use case is running the same benchmark at different concurrency levels to see when the client machine becomes saturated. The `SystemMetricsMonitor` resets between runs, so a single instance can be reused.

In [ ]:
concurrency_levels = [1, 5, 10, 20]
results = {}

for clients in concurrency_levels:
    result = await runner.run(payload=payload, clients=clients, n_requests=5)
    results[clients] = result
    print(
        f"clients={clients:2d}  "
        f"CPU avg={result.stats['system_cpu_percent-average']:5.1f}%  "
        f"RSS max={result.stats['system_memory_rss_mb-max']:6.1f} MB  "
        f"Net recv={result.stats['system_net_bytes_recv_total']:>10,} bytes"
    )

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("CPU Usage vs Concurrency", "Latency vs Concurrency"),
)

clients_list = list(results.keys())

# CPU usage
cpu_avgs = [results[c].stats["system_cpu_percent-average"] for c in clients_list]
cpu_p90s = [results[c].stats["system_cpu_percent-p90"] for c in clients_list]

fig.add_trace(
    go.Scatter(x=clients_list, y=cpu_avgs, mode="lines+markers", name="CPU avg"),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=clients_list, y=cpu_p90s, mode="lines+markers", name="CPU p90"),
    row=1, col=1,
)

# Latency (TTLT p50)
ttlt_p50s = [results[c].stats.get("time_to_last_token-p50", 0) for c in clients_list]
ttlt_p90s = [results[c].stats.get("time_to_last_token-p90", 0) for c in clients_list]

fig.add_trace(
    go.Scatter(x=clients_list, y=ttlt_p50s, mode="lines+markers", name="TTLT p50"),
    row=1, col=2,
)
fig.add_trace(
    go.Scatter(x=clients_list, y=ttlt_p90s, mode="lines+markers", name="TTLT p90"),
    row=1, col=2,
)

fig.update_xaxes(title_text="Concurrent Clients", row=1, col=1)
fig.update_xaxes(title_text="Concurrent Clients", row=1, col=2)
fig.update_yaxes(title_text="CPU %", row=1, col=1)
fig.update_yaxes(title_text="Seconds", row=1, col=2)
fig.update_layout(height=400, title_text="System Resources vs Latency")
fig.show()

## Detecting Client-Side Bottlenecks

After running at multiple concurrency levels, you can check for signs that the client is the bottleneck rather than the endpoint:

- **CPU p90 approaching 100%** — the client process can't keep up with scheduling requests.
- **Memory RSS growing significantly** — asyncio task objects or response data accumulating.
- **Network rates plateauing while latency increases** — possible network saturation.

If you observe these patterns, consider:
- Running LLMeter on a more powerful instance
- Reducing concurrency
- Using `low_memory=True` for very large runs

In [ ]:
# Quick bottleneck check
for clients, result in results.items():
    cpu_p90 = result.stats["system_cpu_percent-p90"]
    rss_max = result.stats["system_memory_rss_mb-max"]
    status = "⚠️  HIGH CPU" if cpu_p90 > 80 else "✅ OK"
    print(f"clients={clients:2d}  CPU p90={cpu_p90:5.1f}%  RSS max={rss_max:6.1f} MB  {status}")

## System-Wide vs Per-Process Monitoring

By default, `SystemMetricsMonitor` tracks the current Python process only (`per_process=True`). Set `per_process=False` to monitor the entire machine — useful when other processes (e.g., a local model server) contribute to the workload.

Note that **network I/O is always system-wide** regardless of this setting, since `psutil` doesn't support per-process network counters on most platforms.

In [ ]:
# System-wide monitoring example
system_monitor = SystemMetricsMonitor(sample_interval=0.5, per_process=False)

system_runner = Runner(
    endpoint=endpoint,
    callbacks=[system_monitor],
    output_path="outputs/system-metrics-wide",
)

result_wide = await system_runner.run(payload=payload, clients=5, n_requests=5)

print(f"System-wide CPU avg: {result_wide.stats['system_cpu_percent-average']:.1f}%")
print(f"System-wide Memory:  {result_wide.stats['system_memory_rss_mb-max']:.1f} MB (used/total)")

## Persistence

System metrics are automatically included in `stats.json` when you save results (via `output_path`). They survive `Result.load()` round-trips, so you can analyze them later without re-running the benchmark.

In [ ]:
from llmeter.results import Result

# Load a previously-saved result
if result.output_path:
    loaded = Result.load(result.output_path)
    print(f"Loaded CPU avg: {loaded.stats['system_cpu_percent-average']:.1f}%")
    print(f"Loaded RSS max: {loaded.stats['system_memory_rss_mb-max']:.1f} MB")

## Summary

The `SystemMetricsMonitor` callback provides visibility into client-side resource usage during benchmarks. Key takeaways:

- **Attach it to any Runner** to get CPU, memory, and network stats for free.
- **Live display** shows real-time metrics during the run.
- **Reusable** across multiple runs — resets automatically.
- **Persistent** — stats survive save/load round-trips in `stats.json`.
- **Raw samples** available for custom time-series analysis.

Use it to validate that your benchmark environment has sufficient headroom, and that latency measurements reflect the endpoint rather than client saturation.